# Data Pipeline

Data cleaning, song clustering (K-means), and user taste profile construction.

Inputs:
- `Original csv/spotify_dataset_gigasheet.csv` — raw dataset
- `playlist/dataset/USETHIS_output_filtered.csv` — user-playlist data

Outputs:
- `spotify_cleaned.csv`
- `K-means/cluster{5,10,15,20}.csv`
- `taste_profiles.csv`

# 1. Data Cleaning


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

In [ ]:
df = pd.read_csv("./Original csv/spotify_dataset_gigasheet.csv")
df

In [ ]:
print((df['Title'] == 'None').sum())
df.isna().sum()

In [ ]:
df['Album_type'].value_counts()

In [ ]:
#remove compilations
df = df[(df['Album_type'] != "compilation")].copy()

In [ ]:
df['Duration_min'].sort_values()

In [ ]:
# remove tracks longer than 10 mins
df = df[(df['Duration_min'] < 10)].copy()

In [ ]:
df = df.sort_values('Stream', ascending=False)
df.duplicated(subset=['Artist', 'Track']).value_counts()

In [ ]:
# remove duplicate artist/track combos
df = df.drop_duplicates(subset=['Artist', 'Track'], keep='first')

In [ ]:
numerical_cols = df.select_dtypes(include='number').columns
zero_counts = (df[numerical_cols] == 0).sum()
print(zero_counts)

In [ ]:
#remove rows with zero tempo and duration
df = df[(df['Tempo'] != 0) & (df['Duration_min'] != 0)].copy()

# set flag for zero views and streams
df['zero_engagement_flag'] = (df['Views'] == 0) | (df['Stream'] == 0)
df

In [ ]:
df.to_csv('spotify_cleaned.csv', index=False)

# 2. K-means Clustering

Cluster songs into K = {5, 10, 15, 20} groups on the 7 audio features.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

In [ ]:
df = pd.read_csv("./spotify_cleaned.csv")
df
df.sort_values(by='Duration_min')
df['Album_type'].value_counts()

In [ ]:
# 7 taste profile features only — aligned with independence_tests.ipynb results
# Dropped: Loudness (redundant with Energy, MI=0.643), Liveness (weak taste signal),
#          Duration_min (not taste), EnergyLiveness (engineered/redundant)
features = [
    'Danceability', 'Energy', 'Valence', 'Acousticness',
    'Instrumentalness', 'Speechiness', 'Tempo'
]
X = df[features]
X

In [ ]:
# scale each column to have mean 0 and sd = 1
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [ ]:
kmeans = KMeans(n_clusters=5, random_state=42)
df['cluster5'] = kmeans.fit_predict(X_scaled)

centers = pd.DataFrame(scaler.inverse_transform(kmeans.cluster_centers_), columns=features)
print(centers)
centers.to_csv('./K-means/cluster5.csv', index=False)

In [ ]:
kmeans = KMeans(n_clusters=10, random_state=42)
df['cluster10'] = kmeans.fit_predict(X_scaled)

centers = pd.DataFrame(scaler.inverse_transform(kmeans.cluster_centers_), columns=features)
print(centers)
centers.to_csv('./K-means/cluster10.csv', index=False)

In [ ]:
kmeans = KMeans(n_clusters=15, random_state=42)
df['cluster15'] = kmeans.fit_predict(X_scaled)

centers = pd.DataFrame(scaler.inverse_transform(kmeans.cluster_centers_), columns=features)
print(centers)
centers.to_csv('./K-means/cluster15.csv', index=False)

In [ ]:
kmeans = KMeans(n_clusters=20, random_state=42)
df['cluster20'] = kmeans.fit_predict(X_scaled)

centers = pd.DataFrame(scaler.inverse_transform(kmeans.cluster_centers_), columns=features)
print(centers)
centers.to_csv('./K-means/cluster20.csv', index=False)

# 3. Taste Profiles

Build a 7-dimensional taste vector for each user by averaging their listened-song features.

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Load datasets
songs = pd.read_csv('spotify_cleaned.csv')
playlists = pd.read_csv('playlist/dataset/USETHIS_output_filtered.csv')

print("=" * 60)
print("STEP 0: LOAD DATA")
print("=" * 60)
print(f"Songs dataset:    {songs.shape[0]} rows, {songs.shape[1]} columns")
print(f"Playlist dataset: {playlists.shape[0]} rows, {playlists.shape[1]} columns")
print(f"Unique playlists: {playlists['pid'].nunique()}")
print(f"Unique songs in playlists: {playlists.groupby(['artist_name','track_name']).ngroups}")
print(f"\nPlaylist columns: {list(playlists.columns)}")
print(f"Songs columns: {list(songs.columns)}")

In [ ]:
TASTE_FEATURES = [
    'Danceability', 'Energy', 'Valence',
    'Acousticness', 'Instrumentalness', 'Speechiness',
    'Tempo'
]

print("=" * 60)
print("STEP 1: DEDUPLICATE PLAYLISTS")
print("=" * 60)
print(f"Before deduplication: {len(playlists)} rows")

# Find duplicates before removing
dupes = playlists[playlists.duplicated(subset=['pid', 'artist_name', 'track_name'], keep='first')]
print(f"Duplicate rows found: {len(dupes)}")
print(f"\nExamples of duplicates removed:")
dupes_sample = dupes.head(10)
for _, row in dupes_sample.iterrows():
    print(f"  pid={row['pid']:>4d}  {row['artist_name']} - {row['track_name']}")

# Remove duplicates
playlists_clean = playlists.drop_duplicates(subset=['pid', 'artist_name', 'track_name'], keep='first')
print(f"\nAfter deduplication: {len(playlists_clean)} rows")
print(f"Rows removed: {len(playlists) - len(playlists_clean)}")

In [ ]:
print("=" * 60)
print("STEP 2: JOIN WITH SONG FEATURES")
print("=" * 60)

# Join on artist + track
joined = playlists_clean.merge(
    songs[['Artist', 'Track'] + TASTE_FEATURES],
    left_on=['artist_name', 'track_name'],
    right_on=['Artist', 'Track'],
    how='left'
)

matched = joined['Artist'].notna().sum()
unmatched = joined['Artist'].isna().sum()

print(f"Matched rows:   {matched} ({matched/len(joined)*100:.2f}%)")
print(f"Unmatched rows: {unmatched} ({unmatched/len(joined)*100:.2f}%)")

if unmatched > 0:
    print(f"\nUnmatched songs:")
    unmatched_songs = joined[joined['Artist'].isna()][['pid', 'artist_name', 'track_name']]
    for _, row in unmatched_songs.iterrows():
        print(f"  pid={row['pid']:>4d}  {row['artist_name']} - {row['track_name']}")

# Drop unmatched rows and extra columns
joined = joined.dropna(subset=['Artist'])
joined = joined.drop(columns=['Artist', 'Track'])

print(f"\nJoined dataset: {len(joined)} rows with {len(TASTE_FEATURES)} taste features")
print(f"Playlists in joined data: {joined['pid'].nunique()}")

In [ ]:
print("=" * 60)
print("STEP 3: FILTER SMALL PLAYLISTS (min 3 songs)")
print("=" * 60)

songs_per_playlist = joined.groupby('pid').size()

print("Playlist size distribution before filtering:")
print(f"  1-2 songs:  {(songs_per_playlist <= 2).sum()} playlists")
print(f"  3-5 songs:  {((songs_per_playlist >= 3) & (songs_per_playlist <= 5)).sum()} playlists")
print(f"  6-10 songs: {((songs_per_playlist >= 6) & (songs_per_playlist <= 10)).sum()} playlists")
print(f"  11+ songs:  {(songs_per_playlist >= 11).sum()} playlists")

# Filter
valid_pids = songs_per_playlist[songs_per_playlist >= 3].index
removed_pids = songs_per_playlist[songs_per_playlist < 3].index

print(f"\nPlaylists removed (< 3 songs): {len(removed_pids)}")
print(f"Playlists remaining: {len(valid_pids)}")

joined_filtered = joined[joined['pid'].isin(valid_pids)]
print(f"Rows remaining: {len(joined_filtered)}")

In [ ]:
print("=" * 60)
print("STEP 4: NORMALIZE TEMPO TO [0, 1]")
print("=" * 60)

tempo_min = joined_filtered['Tempo'].min()
tempo_max = joined_filtered['Tempo'].max()

print(f"Tempo before normalization:")
print(f"  Min: {tempo_min:.3f}")
print(f"  Max: {tempo_max:.3f}")
print(f"  Mean: {joined_filtered['Tempo'].mean():.3f}")

joined_filtered = joined_filtered.copy()
joined_filtered['Tempo'] = (joined_filtered['Tempo'] - tempo_min) / (tempo_max - tempo_min)

print(f"\nTempo after normalization:")
print(f"  Min: {joined_filtered['Tempo'].min():.3f}")
print(f"  Max: {joined_filtered['Tempo'].max():.3f}")
print(f"  Mean: {joined_filtered['Tempo'].mean():.3f}")

print(f"\nAll feature ranges after normalization:")
for feat in TASTE_FEATURES:
    print(f"  {feat:20s}  [{joined_filtered[feat].min():.3f}, {joined_filtered[feat].max():.3f}]")

In [ ]:
print("=" * 60)
print("STEP 5: BUILD TASTE PROFILES (mean per user)")
print("=" * 60)

taste_profiles = joined_filtered.groupby('pid')[TASTE_FEATURES].mean()

print(f"Taste profiles built: {len(taste_profiles)} users")
print(f"Features per profile: {len(TASTE_FEATURES)}")
print(f"\nTaste profile statistics across all users:")
print(taste_profiles.describe().round(4))

print(f"\nExample profiles:")
examples = taste_profiles.head(5)
for pid, row in examples.iterrows():
    n_songs = songs_per_playlist[pid]
    print(f"\n  User (pid={pid}), {n_songs} songs:")
    for feat in TASTE_FEATURES:
        bar = '#' * int(row[feat] * 30)
        print(f"    {feat:20s}  {row[feat]:.3f}  {bar}")

In [ ]:
print("=" * 60)
print("STEP 6: SAVE")
print("=" * 60)

# Save taste profiles
taste_profiles.to_csv('taste_profiles.csv')
print(f"Saved taste_profiles.csv ({len(taste_profiles)} users x {len(TASTE_FEATURES)} features)")

print(f"\nTempo normalization params (apply same scaling to candidate songs):")
print(f"  min={tempo_min:.3f}, max={tempo_max:.3f}")

print(f"\nFinal output:")
print(taste_profiles.head(10).round(4))